# ICT NY FVG 策略探索

本 Notebook 用於研究和驗證 ICT (Inner Circle Trader) NY Session FVG (Fair Value Gap) 策略

## 策略概述
- **時間框架**: 5分鐘
- **交易時段**: NY Session (09:30-11:30 EST)
- **核心邏輯**: Liquidity Sweep + Fair Value Gap + 1:3 風險收益比


In [ ]:
# 導入必要的套件
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, time
import pytz

# 設定繪圖樣式
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# 導入自定義模組
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from core.strategy_ict_ny_fvg import ICTNYFVGStrategy
from core.strategy_base import TradingSignal, SignalType, BarData

## 1. 策略初始化

In [ ]:
# 策略配置
strategy_config = {
    "name": "ICT_NY_FVG_Research",
    "parameters": {
        "session_start": "09:30",
        "session_end": "11:30", 
        "timezone": "America/New_York",
        "min_fvg_size_pips": 10,
        "max_fvg_size_pips": 100,
        "fvg_timeout_bars": 10,
        "lookback_bars": 5,
        "risk_reward_ratio": 3.0,
        "stop_loss_buffer_pips": 2,
        "take_profit_buffer_pips": 1
    }
}

# 初始化策略
strategy = ICTNYFVGStrategy(strategy_config)
print(f"策略初始化完成: {strategy.name}")
print(f"策略參數: {strategy.parameters}")

## 2. 生成模擬數據

創建符合 NY Session 時間的模擬 BTCUSDT 5分鐘數據

In [ ]:
def generate_mock_data(start_date='2024-01-01', days=30):
    """生成模擬的加密貨幣價格數據"""
    
    # 生成 NY Session 時間範圍的數據
    ny_tz = pytz.timezone('America/New_York')
    dates = []
    
    for day in pd.date_range(start_date, periods=days, freq='D'):
        # 每天 09:30-11:30 的 5分鐘 K線
        session_start = ny_tz.localize(datetime.combine(day.date(), time(9, 30)))
        session_end = ny_tz.localize(datetime.combine(day.date(), time(11, 30)))
        session_times = pd.date_range(session_start, session_end, freq='5min')
        dates.extend(session_times)
    
    # 生成價格數據（隨機遊走 + 趨勢）
    np.random.seed(42)
    n_bars = len(dates)
    
    # 基準價格
    base_price = 45000
    
    # 價格變動（隨機遊走）
    returns = np.random.normal(0, 0.002, n_bars)  # 0.2% 標準差
    
    # 添加一些趨勢和波動性
    for i in range(1, len(returns)):
        # 增加一些自相關性
        returns[i] += returns[i-1] * 0.1
        
        # 隨機添加大波動
        if np.random.rand() < 0.05:  # 5% 機率
            returns[i] += np.random.choice([-1, 1]) * 0.01  # ±1% 突然變動
    
    # 計算累積價格
    prices = base_price * np.exp(np.cumsum(returns))
    
    # 生成 OHLCV 數據
    data = []
    for i, (timestamp, close_price) in enumerate(zip(dates, prices)):
        # 上一根收盤作為這根開盤
        open_price = prices[i-1] if i > 0 else close_price
        
        # 高低點（基於開盤和收盤）
        high_low_range = abs(close_price - open_price) * 2 + np.random.uniform(5, 50)
        high = max(open_price, close_price) + np.random.uniform(0, high_low_range * 0.5)
        low = min(open_price, close_price) - np.random.uniform(0, high_low_range * 0.5)
        
        volume = np.random.uniform(100, 2000)
        
        data.append({
            'timestamp': timestamp,
            'open': open_price,
            'high': high,
            'low': low, 
            'close': close_price,
            'volume': volume
        })
    
    df = pd.DataFrame(data)
    df.set_index('timestamp', inplace=True)
    
    return df

# 生成測試數據
test_data = generate_mock_data(days=10)
print(f"生成 {len(test_data)} 根 K線數據")
print(f"時間範圍: {test_data.index[0]} 至 {test_data.index[-1]}")
print(f"價格範圍: ${test_data['low'].min():.2f} - ${test_data['high'].max():.2f}")

# 顯示前幾筆數據
test_data.head()

## 3. 視覺化價格數據

In [ ]:
# 繪製價格圖表
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# 收盤價時序圖
ax1.plot(test_data.index, test_data['close'], label='收盤價', linewidth=1)
ax1.fill_between(test_data.index, test_data['low'], test_data['high'], 
                 alpha=0.3, label='高低價範圍')
ax1.set_title('BTCUSDT 5分鐘價格數據 (模擬)', fontsize=14)
ax1.set_ylabel('價格 (USD)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 成交量
ax2.bar(test_data.index, test_data['volume'], alpha=0.7, color='orange')
ax2.set_title('成交量', fontsize=14)
ax2.set_ylabel('成交量', fontsize=12)
ax2.set_xlabel('時間', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 顯示統計資訊
print("數據統計:")
print(test_data.describe())

## 4. 策略測試

測試 ICT NY FVG 策略在模擬數據上的表現

In [ ]:
def test_strategy_on_data(strategy, data, min_history=20):
    """在歷史數據上測試策略"""
    
    signals = []
    
    for i in range(min_history, len(data)):
        # 當前 K線
        current_bar = data.iloc[i]
        bar_data = BarData(
            timestamp=current_bar.name,
            open=current_bar['open'],
            high=current_bar['high'],
            low=current_bar['low'],
            close=current_bar['close'],
            volume=current_bar['volume']
        )
        
        # 歷史數據
        history = data.iloc[:i]
        
        # 獲取交易信號
        signal = strategy.on_bar(bar_data, history)
        
        if signal and strategy.validate_signal(signal):
            signals.append({
                'timestamp': current_bar.name,
                'signal_type': signal.signal_type.value,
                'entry_price': signal.entry_price,
                'stop_loss': signal.stop_loss,
                'take_profit': signal.take_profit,
                'confidence': signal.confidence,
                'reason': signal.reason
            })
            
            print(f"信號產生: {current_bar.name}")
            print(f"  類型: {signal.signal_type.value}")
            print(f"  進場: ${signal.entry_price:.2f}")
            print(f"  止損: ${signal.stop_loss:.2f}")
            print(f"  止盈: ${signal.take_profit:.2f}")
            print(f"  理由: {signal.reason}")
            print("-" * 50)
    
    return pd.DataFrame(signals)

# 運行策略測試
print("開始策略測試...")
signals_df = test_strategy_on_data(strategy, test_data)

print(f"\n測試完成！共產生 {len(signals_df)} 個交易信號")

if len(signals_df) > 0:
    print("\n信號摘要:")
    print(signals_df.groupby('signal_type').size())
    
    display(signals_df)
else:
    print("未產生任何交易信號。可能原因:")
    print("1. 數據中沒有符合 ICT 條件的設置")
    print("2. 策略參數過於嚴格")
    print("3. 模擬數據的特性不適合該策略")

## 5. 策略元件分析

分別測試策略的各個組成部分

In [ ]:
# 測試 NY Session 檢測
print("NY Session 檢測測試:")
for i in range(0, min(10, len(test_data))):
    timestamp = test_data.index[i]
    is_ny_session = strategy._is_ny_session(timestamp)
    print(f"{timestamp}: {is_ny_session}")

print("\n" + "="*50)

# 測試 Liquidity Sweep 偵測
print("Liquidity Sweep 偵測測試:")
for i in range(10, min(30, len(test_data))):
    history = test_data.iloc[:i+1]
    sweep = strategy._detect_liquidity_sweep(history)
    if sweep:
        print(f"時間: {test_data.index[i]}, Sweep: {sweep}")

print("\n" + "="*50)

# 測試 Engulfing Pattern 偵測
print("Engulfing Pattern 偵測測試:")
engulfing_count = 0
for i in range(1, min(50, len(test_data))):
    history = test_data.iloc[:i+1]
    engulfing = strategy._detect_engulfing_pattern(history)
    if engulfing:
        engulfing_count += 1
        print(f"時間: {test_data.index[i]}, Engulfing: {engulfing}")
        if engulfing_count >= 5:  # 只顯示前5個
            break

print("\n" + "="*50)

# 測試 Fair Value Gap 偵測
print("Fair Value Gap 偵測測試:")
fvg_count = 0
for i in range(3, min(100, len(test_data))):
    history = test_data.iloc[:i+1]
    
    # 測試多頭 FVG
    fvg_bull = strategy._detect_fair_value_gap(history, "BULLISH")
    if fvg_bull:
        fvg_count += 1
        print(f"時間: {test_data.index[i]}, Bullish FVG: {fvg_bull['size_pips']:.1f} pips")
    
    # 測試空頭 FVG
    fvg_bear = strategy._detect_fair_value_gap(history, "BEARISH")
    if fvg_bear:
        fvg_count += 1
        print(f"時間: {test_data.index[i]}, Bearish FVG: {fvg_bear['size_pips']:.1f} pips")
    
    if fvg_count >= 5:  # 只顯示前5個
        break

print(f"\n策略元件測試完成！")

## 6. 策略參數優化建議

基於測試結果提供參數調整建議

In [ ]:
# 分析當前參數設置的合理性
print("策略參數分析:")
print("="*50)

current_params = strategy.parameters

# 分析 FVG 大小設置
print(f"FVG 大小範圍: {current_params['min_fvg_size_pips']} - {current_params['max_fvg_size_pips']} pips")
avg_price = test_data['close'].mean()
min_fvg_usd = current_params['min_fvg_size_pips'] / 10000 * avg_price
max_fvg_usd = current_params['max_fvg_size_pips'] / 10000 * avg_price
print(f"對應美元金額: ${min_fvg_usd:.2f} - ${max_fvg_usd:.2f}")

# 分析風險收益比
print(f"\n風險收益比: 1:{current_params['risk_reward_ratio']}")
print("這意味著每承擔 $1 風險，預期獲得 ${:.2f} 收益".format(current_params['risk_reward_ratio']))

# 分析交易時段
print(f"\n交易時段: {current_params['session_start']} - {current_params['session_end']} ({current_params['timezone']})")
session_duration = 2 * 60 // 5  # 2小時轉換為5分鐘K線數量
print(f"每日可用K線數量: {session_duration} 根")

print("\n" + "="*50)
print("優化建議:")
print("1. 如果信號過少，可以:")
print("   - 放寬 FVG 大小限制 (減少 min_fvg_size_pips)")
print("   - 減少 lookback_bars 以捕捉更多 sweep")
print("   - 擴大交易時段")

print("\n2. 如果信號過多，可以:")
print("   - 提高 FVG 大小要求")
print("   - 增加 lookback_bars 以要求更強的 sweep")
print("   - 縮小交易時段至最活躍時間")

print("\n3. 風險控制:")
print("   - 當前 1:3 風險收益比較為激進")
print("   - 建議先從 1:2 開始測試")
print("   - 可考慮動態調整止盈目標")

## 7. 下一步開發計劃

In [ ]:
print("ICT NY FVG 策略研究總結:")
print("="*50)

print("✅ 已完成:")
print("1. 策略核心邏輯實作")
print("2. 模組化架構設計")
print("3. 基本功能測試")
print("4. 參數化配置")

print("\n🔄 進行中:")
print("1. 策略邏輯驗證與優化")
print("2. 更真實的測試數據")

print("\n📋 待辦事項:")
print("1. 【高優先級】")
print("   - 實作回測引擎")
print("   - 下載真實歷史數據")
print("   - 績效評估指標")

print("\n2. 【中優先級】")
print("   - 交易所 API 整合")
print("   - 模擬交易功能")
print("   - 風險管理優化")

print("\n3. 【低優先級】")
print("   - Web 界面開發")
print("   - 多策略組合")
print("   - 機器學習優化")

print("\n💡 建議下一個 Notebook:")
print("02_data_collection.ipynb - 歷史數據收集與清理")